In [2]:
!pip install -q pyarabic

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import re
from typing import List, Tuple
import os

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("="*70)
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("="*70)

Device: cuda
GPU: Tesla T4
Memory: 15.83 GB


In [ ]:
# ============================================================================
# Configuration
# ============================================================================

DATA_PATH = '/kaggle/input/diacritics/dataset/'

CONFIG = {
    # Data paths
    'train_path': f'{DATA_PATH}train.txt',
    'dev_path': f'{DATA_PATH}val.txt',
    
    # Feature types (3+ required by project)
    'use_char_embeddings': True,      # Feature 1: Trainable character embeddings
    'use_positional_encoding': True,  # Feature 2: Position in sequence
    'use_char_type': True,            # Feature 3: Character type (letter/space/etc)
    'use_contextual': False,          # Feature 4: Contextual embeddings (optional)
    
    # Model hyperparameters
    'embedding_dim': 256,
    'positional_dim': 64,
    'char_type_dim': 32,
    'hidden_dim': 512,
    'num_layers': 3,
    'dropout': 0.3,
    'num_heads': 8,  # For attention mechanism
    
    # Training settings
    'batch_size': 32,
    'num_epochs': 15,
    'learning_rate': 0.001,
    'max_length': 512,
    
    # Model selection
    'train_bilstm': True,
    'train_transformer': True,
}

# Verify data exists
print("Checking data availability...")
print(f"📁 Dataset location: {DATA_PATH}")
print(f"✓ Train file exists: {os.path.exists(CONFIG['train_path'])}")
print(f"✓ Dev file exists: {os.path.exists(CONFIG['dev_path'])}")

# Show available files in input directory
print("\n📂 Available files in /kaggle/input/:")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(f"  {os.path.join(dirname, filename)}")

# Feature summary
print("\n🎯 Features enabled (3+ required):")
features_count = sum([
    CONFIG['use_char_embeddings'],
    CONFIG['use_positional_encoding'],
    CONFIG['use_char_type'],
    CONFIG['use_contextual']
])
print(f"  ✓ Character Embeddings: {CONFIG['use_char_embeddings']}")
print(f"  ✓ Positional Encoding: {CONFIG['use_positional_encoding']}")
print(f"  ✓ Character Type: {CONFIG['use_char_type']}")
print(f"  ✓ Contextual Embeddings: {CONFIG['use_contextual']}")
print(f"  📊 Total features: {features_count} {'✅' if features_count >= 3 else '❌ Need at least 3!'}")

In [ ]:
# ============================================================================
# Diacritic Definitions
# ============================================================================

DIACRITIC_TO_ID = {
    '': 0,   # No diacritic
    'َ': 1,  # Fatha
    'ً': 2,  # Tanween Fath
    'ُ': 3,  # Damma
    'ٌ': 4,  # Tanween Damm
    'ِ': 5,  # Kasra
    'ٍ': 6,  # Tanween Kasr
    'ْ': 7,  # Sukun
    'ّ': 8,  # Shadda
}

ID_TO_DIACRITIC = {v: k for k, v in DIACRITIC_TO_ID.items()}
NUM_CLASSES = len(DIACRITIC_TO_ID)

DIACRITICS = {
    'َ': 'Fatha', 'ً': 'Tanween Fath', 'ُ': 'Damma',
    'ٌ': 'Tanween Damm', 'ِ': 'Kasra', 'ٍ': 'Tanween Kasr',
    'ْ': 'Sukun', 'ّ': 'Shadda', '': 'No Diacritic'
}

print(f"✓ Number of diacritic classes: {NUM_CLASSES}")

In [ ]:
# ============================================================================
# Preprocessing Class
# ============================================================================

class ArabicDataPreprocessor:
    def __init__(self):
        self.arabic_letters = set('ابتثجحخدذرزسشصضطظعغفقكلمنهويءآأؤإئةى')
        self.diacritics = set(DIACRITIC_TO_ID.keys()) - {''}
        
    def clean_text(self, text: str) -> str:
        """Clean text while preserving Arabic and diacritics"""
        # Remove HTML tags
        text = re.sub(r'<[^>]+>', '', text)
        # Remove English letters and numbers
        text = re.sub(r'[a-zA-Z0-9]+', '', text)
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        # Remove Tatweel (elongation)
        text = text.replace('ـ', '')
        return text
    
    def extract_diacritics(self, text: str) -> Tuple[str, List[str]]:
        """
        Separate diacritized text into:
        1. Undiacritized text (input for model)
        2. List of diacritics per character (labels for model)
        """
        undiacritized = []
        diacritics_list = []
        
        for char in text:
            if char in self.arabic_letters or char == ' ':
                undiacritized.append(char)
                diacritics_list.append('')  # No diacritic initially
            elif char in self.diacritics:
                if diacritics_list:
                    # Attach diacritic to previous character
                    diacritics_list[-1] = char
                    
        return ''.join(undiacritized), diacritics_list
    
    def load_dataset(self, filepath: str) -> List[Tuple[str, str, List[str]]]:
        """
        Load diacritized dataset
        Returns: List of (original_text, undiacritized_text, diacritic_labels)
        """
        dataset = []
        with open(filepath, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                
                # Clean the text
                cleaned = self.clean_text(line)
                
                # Extract undiacritized text and diacritic labels
                undiacritized, labels = self.extract_diacritics(cleaned)
                
                if undiacritized and labels:
                    dataset.append((cleaned, undiacritized, labels))
        
        return dataset
    
    def calculate_diacritic_stats(self, dataset):
        """Calculate and display diacritic distribution"""
        diac_counts = {d: 0 for d in DIACRITIC_TO_ID.keys()}
        total_chars = 0
        
        for _, _, labels in dataset:
            for label in labels:
                if label in diac_counts:
                    diac_counts[label] += 1
                total_chars += 1
        
        print("\n📊 Diacritic Distribution:")
        print("-" * 50)
        for diac, count in sorted(diac_counts.items(), key=lambda x: x[1], reverse=True):
            name = DIACRITICS.get(diac, 'No Diacritic')
            pct = (count / total_chars) * 100 if total_chars > 0 else 0
            print(f"{name:20s}: {count:8d} ({pct:5.2f}%)")
        print("-" * 50)

print("✓ Preprocessing class defined")

In [ ]:

# ============================================================================
# Character Vocabulary
# ============================================================================

class CharacterVocabulary:
    """Build vocabulary for character-level modeling"""
    def __init__(self):
        self.char2id = {'<PAD>': 0, '<UNK>': 1}
        self.id2char = {0: '<PAD>', 1: '<UNK>'}
        
        # Add all Arabic letters and space
        arabic_chars = 'ابتثجحخدذرزسشصضطظعغفقكلمنهويءآأؤإئةى '
        for char in arabic_chars:
            if char not in self.char2id:
                idx = len(self.char2id)
                self.char2id[char] = idx
                self.id2char[idx] = char
    
    def encode(self, text: str) -> List[int]:
        """Convert text to character IDs"""
        return [self.char2id.get(c, self.char2id['<UNK>']) for c in text]
    
    def decode(self, ids: List[int]) -> str:
        """Convert character IDs back to text"""
        return ''.join([self.id2char.get(i, '<UNK>') for i in ids])
    
    def __len__(self):
        return len(self.char2id)

print("✓ Vocabulary class defined")


In [ ]:
# ============================================================================
# CELL 6: PyTorch Dataset Class (WITH 3+ FEATURES)
# ============================================================================

class DiacritizationDataset(Dataset):
    """PyTorch Dataset with multiple feature types"""
    
    def __init__(self, data, char_vocab, diacritic_vocab, max_length=512):
        self.data = data
        self.char_vocab = char_vocab
        self.diacritic_vocab = diacritic_vocab
        self.max_length = max_length
        
        # Character type mapping
        self.char_type_map = self._build_char_type_map()
        
    def _build_char_type_map(self):
        """Map characters to types: letter, space, punctuation, etc."""
        char_types = {}
        
        # Arabic letters
        arabic_letters = set('ابتثجحخدذرزسشصضطظعغفقكلمنهويءآأؤإئةى')
        for char in arabic_letters:
            char_types[char] = 1  # Type 1: Letter
        
        # Space
        char_types[' '] = 2  # Type 2: Space
        
        # Punctuation (if any)
        for char in '.,;:!?':
            char_types[char] = 3  # Type 3: Punctuation
        
        # Default
        char_types['<PAD>'] = 0
        char_types['<UNK>'] = 4
        
        return char_types
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        original, undiacritized, labels = self.data[idx]
        
        # Feature 1: Character IDs (for trainable embeddings)
        char_ids = self.char_vocab.encode(undiacritized)
        
        # Feature 2: Positional encoding (position in sequence)
        positions = list(range(len(char_ids)))
        
        # Feature 3: Character type features
        char_types = [self.char_type_map.get(c, 4) for c in undiacritized]
        
        # Encode diacritics to IDs
        label_ids = [self.diacritic_vocab.get(l, 0) for l in labels]
        
        # Truncate if too long
        if len(char_ids) > self.max_length:
            char_ids = char_ids[:self.max_length]
            positions = positions[:self.max_length]
            char_types = char_types[:self.max_length]
            label_ids = label_ids[:self.max_length]
        
        # Pad to max_length
        padding_length = self.max_length - len(char_ids)
        char_ids += [0] * padding_length
        positions += [0] * padding_length
        char_types += [0] * padding_length
        label_ids += [0] * padding_length
        
        # Create attention mask (1 for real tokens, 0 for padding)
        attention_mask = [1] * (self.max_length - padding_length) + [0] * padding_length
        
        return {
            'char_ids': torch.tensor(char_ids, dtype=torch.long),
            'positions': torch.tensor(positions, dtype=torch.long),
            'char_types': torch.tensor(char_types, dtype=torch.long),
            'labels': torch.tensor(label_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        }

print("✓ Dataset class defined (with 3 feature types)")

In [ ]:
# ============================================================================
# BiLSTM Model (WITH 3+ FEATURES + ATTENTION)
# ============================================================================

class EnhancedBiLSTMDiacritizer(nn.Module):
    """
    Enhanced BiLSTM with:
    - 3+ feature types (char embeddings, positional, char-type)
    - Multi-head attention
    - Layer normalization
    - Residual connections
    """
    
    def __init__(self, vocab_size, embedding_dim=256, positional_dim=64,
                 char_type_dim=32, hidden_dim=512, num_layers=3, 
                 num_classes=9, dropout=0.3, num_heads=8):
        super().__init__()
        
        # Feature 1: Character embeddings (trainable)
        self.char_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Feature 2: Positional embeddings
        self.pos_embedding = nn.Embedding(512, positional_dim, padding_idx=0)
        
        # Feature 3: Character type embeddings
        self.char_type_embedding = nn.Embedding(5, char_type_dim, padding_idx=0)
        
        # Total embedding dimension
        total_embedding_dim = embedding_dim + positional_dim + char_type_dim
        
        # Layer normalization
        self.layer_norm1 = nn.LayerNorm(total_embedding_dim)
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            total_embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        self.dropout = nn.Dropout(dropout)
        
        # Multi-head attention for context
        lstm_output_dim = hidden_dim * 2  # Bidirectional
        self.attention = nn.MultiheadAttention(
            lstm_output_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )
        
        self.layer_norm2 = nn.LayerNorm(lstm_output_dim)
        
        # Classification head with residual connection
        self.classifier = nn.Sequential(
            nn.Linear(lstm_output_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
        
    def forward(self, char_ids, positions, char_types, attention_mask=None):
        # Feature 1: Character embeddings
        char_emb = self.char_embedding(char_ids)
        
        # Feature 2: Positional embeddings
        pos_emb = self.pos_embedding(positions)
        
        # Feature 3: Character type embeddings
        type_emb = self.char_type_embedding(char_types)
        
        # Concatenate all features
        embedded = torch.cat([char_emb, pos_emb, type_emb], dim=-1)
        embedded = self.layer_norm1(embedded)
        embedded = self.dropout(embedded)
        
        # BiLSTM encoding
        lstm_out, _ = self.lstm(embedded)
        lstm_out = self.dropout(lstm_out)
        
        # Multi-head attention (with residual connection)
        if attention_mask is not None:
            # Convert attention mask to proper format for MultiheadAttention
            # (1 -> can attend, 0 -> cannot attend)
            key_padding_mask = (attention_mask == 0)
        else:
            key_padding_mask = None
        
        attn_out, _ = self.attention(
            lstm_out, lstm_out, lstm_out,
            key_padding_mask=key_padding_mask
        )
        
        # Residual connection + layer norm
        lstm_out = self.layer_norm2(lstm_out + attn_out)
        lstm_out = self.dropout(lstm_out)
        
        # Classification
        logits = self.classifier(lstm_out)
        
        return logits

print("✓ Enhanced BiLSTM architecture defined (3 features + attention)")

In [ ]:
# ============================================================================
# Training Functions
# ============================================================================

def train_epoch(model, train_loader, optimizer, scheduler, criterion, device):
    """Train for one epoch with multiple features"""
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc='Training')
    
    for batch in progress_bar:
        char_ids = batch['char_ids'].to(device)
        positions = batch['positions'].to(device)
        char_types = batch['char_types'].to(device)
        labels = batch['labels'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        
        # Check if model is Transformer or BiLSTM
        if isinstance(model, TransformerDiacritizer):
            logits = model(char_ids, positions, char_types, attention_mask)
        else:
            logits = model(char_ids, positions, char_types, attention_mask)
        
        # Calculate loss
        loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(train_loader)

def validate(model, val_loader, criterion, device):
    """Validate the model with multiple features"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Validation'):
            char_ids = batch['char_ids'].to(device)
            positions = batch['positions'].to(device)
            char_types = batch['char_types'].to(device)
            labels = batch['labels'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            # Forward pass
            if isinstance(model, TransformerDiacritizer):
                logits = model(char_ids, positions, char_types, attention_mask)
            else:
                logits = model(char_ids, positions, char_types, attention_mask)
            
            loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))
            total_loss += loss.item()
            
            # Get predictions
            preds = torch.argmax(logits, dim=-1)
            
            # Only consider non-padded positions
            mask = attention_mask.bool()
            all_preds.extend(preds[mask].cpu().numpy())
            all_labels.extend(labels[mask].cpu().numpy())
    
    avg_loss = total_loss / len(val_loader)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Calculate metrics
    errors = (all_preds != all_labels).sum()
    total = len(all_labels)
    der = errors / total if total > 0 else 0
    accuracy = 1 - der
    
    return avg_loss, accuracy, der

print("✓ Training functions updated for multiple features")

In [ ]:
# ============================================================================
# Prediction Function (UPDATED FOR NEW FEATURES)
# ============================================================================

def predict_diacritics(model, text, char_vocab, id_to_diacritic, device, max_length=512):
    """Predict diacritics for input text with multiple features"""
    model.eval()
    
    # Build char type map
    char_type_map = {}
    arabic_letters = set('ابتثجحخدذرزسشصضطظعغفقكلمنهويءآأؤإئةى')
    for char in arabic_letters:
        char_type_map[char] = 1
    char_type_map[' '] = 2
    for char in '.,;:!?':
        char_type_map[char] = 3
    
    # Encode text
    char_ids = char_vocab.encode(text)
    positions = list(range(len(char_ids)))
    char_types = [char_type_map.get(c, 4) for c in text]
    
    if len(char_ids) > max_length:
        char_ids = char_ids[:max_length]
        positions = positions[:max_length]
        char_types = char_types[:max_length]
    
    # Add batch dimension and move to device
    char_ids = torch.tensor([char_ids], dtype=torch.long).to(device)
    positions = torch.tensor([positions], dtype=torch.long).to(device)
    char_types = torch.tensor([char_types], dtype=torch.long).to(device)
    
    # Predict
    with torch.no_grad():
        # Check if model is Transformer or BiLSTM
        if isinstance(model, TransformerDiacritizer):
            logits = model(char_ids, positions, char_types)
        else:
            logits = model(char_ids, positions, char_types)
        preds = torch.argmax(logits, dim=-1)
    
    # Decode predictions
    pred_ids = preds[0].cpu().numpy()
    diacritics = [id_to_diacritic.get(int(p), '') for p in pred_ids[:len(text)]]
    
    # Combine text with diacritics
    result = ''
    for char, diac in zip(text, diacritics):
        result += char + diac
    
    return result

print("✓ Prediction function updated for multiple features")

In [ ]:

# ============================================================================
# Load and Prepare Data
# ============================================================================

print("\n" + "="*70)
print("📚 LOADING DATA")
print("="*70)

preprocessor = ArabicDataPreprocessor()

print("\n⏳ Loading training data...")
train_data = preprocessor.load_dataset(CONFIG['train_path'])
print(f"✓ Loaded {len(train_data)} training samples")

print("\n⏳ Loading validation data...")
dev_data = preprocessor.load_dataset(CONFIG['dev_path'])
print(f"✓ Loaded {len(dev_data)} validation samples")

# Show data statistics
preprocessor.calculate_diacritic_stats(train_data)

# Example of what the data looks like
if train_data:
    print("\n📝 Example from training data:")
    original, undiacritized, labels = train_data[0]
    print(f"  Original (with diacritics): {original[:80]}...")
    print(f"  Undiacritized (model input): {undiacritized[:80]}...")
    print(f"  First 10 labels: {labels[:10]}")

In [ ]:
# ============================================================================
# Create Vocabulary and Datasets
# ============================================================================

print("\n" + "="*70)
print("🔤 BUILDING VOCABULARY AND DATASETS")
print("="*70)

# Create character vocabulary
char_vocab = CharacterVocabulary()
print(f"\n✓ Character vocabulary size: {len(char_vocab)}")

# Create PyTorch datasets
print("\n⏳ Creating PyTorch datasets...")
train_dataset = DiacritizationDataset(
    train_data, char_vocab, DIACRITIC_TO_ID, CONFIG['max_length']
)
dev_dataset = DiacritizationDataset(
    dev_data, char_vocab, DIACRITIC_TO_ID, CONFIG['max_length']
)
print(f"✓ Train dataset: {len(train_dataset)} samples")
print(f"✓ Dev dataset: {len(dev_dataset)} samples")

# Create dataloaders
print("\n⏳ Creating dataloaders...")
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
dev_loader = DataLoader(
    dev_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)
print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Dev batches: {len(dev_loader)}")

In [ ]:
# ============================================================================
# Initialize Model
# ============================================================================

print("\n" + "="*70)
print("🤖 INITIALIZING MODEL")
print("="*70)

model = EnhancedBiLSTMDiacritizer(
    vocab_size=len(char_vocab),
    embedding_dim=CONFIG['embedding_dim'],
    positional_dim=CONFIG['positional_dim'],
    char_type_dim=CONFIG['char_type_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    num_layers=CONFIG['num_layers'],
    num_classes=NUM_CLASSES,
    dropout=CONFIG['dropout'],
    num_heads=CONFIG['num_heads']
)

model = model.to(device)
num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n✓ Model: Enhanced BiLSTM with 3+ Features")
print(f"  - Features:")
print(f"    • Character embeddings: {CONFIG['embedding_dim']}d")
print(f"    • Positional encoding: {CONFIG['positional_dim']}d")
print(f"    • Character type: {CONFIG['char_type_dim']}d")
print(f"  - Hidden dim: {CONFIG['hidden_dim']}")
print(f"  - Num layers: {CONFIG['num_layers']}")
print(f"  - Attention heads: {CONFIG['num_heads']}")
print(f"  - Dropout: {CONFIG['dropout']}")
print(f"  - Total parameters: {num_params:,}")
print(f"  - Trainable parameters: {trainable_params:,}")

In [ ]:
# ============================================================================
# Transformer Model (2ND MODEL)
# ============================================================================

class TransformerDiacritizer(nn.Module):
    """
    Transformer model for Arabic diacritization
    (2nd model as required by project)
    """
    
    def __init__(self, vocab_size, embedding_dim=256, positional_dim=64,
                 char_type_dim=32, hidden_dim=512, num_layers=4,
                 num_classes=9, dropout=0.3, num_heads=8):
        super().__init__()
        
        # Feature embeddings (same 3 features as BiLSTM)
        self.char_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(512, positional_dim, padding_idx=0)
        self.char_type_embedding = nn.Embedding(5, char_type_dim, padding_idx=0)
        
        total_embedding_dim = embedding_dim + positional_dim + char_type_dim
        
        # Project to hidden dim
        self.input_projection = nn.Linear(total_embedding_dim, hidden_dim)
        
        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
        
    def forward(self, char_ids, positions, char_types, attention_mask=None):
        # Concatenate all features
        char_emb = self.char_embedding(char_ids)
        pos_emb = self.pos_embedding(positions)
        type_emb = self.char_type_embedding(char_types)
        
        embedded = torch.cat([char_emb, pos_emb, type_emb], dim=-1)
        embedded = self.input_projection(embedded)
        embedded = self.dropout(embedded)
        
        # Create padding mask for transformer
        if attention_mask is not None:
            src_key_padding_mask = (attention_mask == 0)
        else:
            src_key_padding_mask = None
        
        # Transformer encoding
        transformer_out = self.transformer(
            embedded,
            src_key_padding_mask=src_key_padding_mask
        )
        
        transformer_out = self.layer_norm(transformer_out)
        transformer_out = self.dropout(transformer_out)
        
        # Classification
        logits = self.classifier(transformer_out)
        
        return logits

print("✓ Transformer model defined (2nd model)")

In [ ]:
# ============================================================================
# TRAIN BOTH MODELS AND SELECT BEST
# ============================================================================

print("\n" + "="*70)
print("🏆 TRAINING BOTH MODELS (BiLSTM + Transformer)")
print("="*70)

models_to_train = []

# Model 1: Enhanced BiLSTM
if CONFIG['train_bilstm']:
    bilstm_model = EnhancedBiLSTMDiacritizer(
        vocab_size=len(char_vocab),
        embedding_dim=CONFIG['embedding_dim'],
        positional_dim=CONFIG['positional_dim'],
        char_type_dim=CONFIG['char_type_dim'],
        hidden_dim=CONFIG['hidden_dim'],
        num_layers=CONFIG['num_layers'],
        num_classes=NUM_CLASSES,
        dropout=CONFIG['dropout'],
        num_heads=CONFIG['num_heads']
    ).to(device)
    models_to_train.append(('BiLSTM', bilstm_model))

# Model 2: Transformer
if CONFIG['train_transformer']:
    transformer_model = TransformerDiacritizer(
        vocab_size=len(char_vocab),
        embedding_dim=CONFIG['embedding_dim'],
        positional_dim=CONFIG['positional_dim'],
        char_type_dim=CONFIG['char_type_dim'],
        hidden_dim=CONFIG['hidden_dim'],
        num_layers=4,
        num_classes=NUM_CLASSES,
        dropout=CONFIG['dropout'],
        num_heads=CONFIG['num_heads']
    ).to(device)
    models_to_train.append(('Transformer', transformer_model))

print(f"\n📋 Training {len(models_to_train)} models:")
for name, _ in models_to_train:
    print(f"  • {name}")

# Track best model
best_overall_der = float('inf')
best_model_name = None
best_model_state = None

# Train each model
for model_name, model in models_to_train:
    print(f"\n{'='*70}")
    print(f"🚀 Training {model_name}")
    print(f"{'='*70}")
    
    # Setup optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=0.01)
    total_steps = len(train_loader) * CONFIG['num_epochs']
    scheduler = OneCycleLR(
        optimizer,
        max_lr=CONFIG['learning_rate'],
        total_steps=total_steps,
        pct_start=0.1,
        anneal_strategy='cos'
    )
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    
    best_der = float('inf')
    
    for epoch in range(CONFIG['num_epochs']):
        print(f"\n📍 Epoch {epoch + 1}/{CONFIG['num_epochs']}")
        
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, device)
        
        # Validate
        val_loss, val_acc, val_der = validate(model, dev_loader, criterion, device)
        
        print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"  Val Acc: {val_acc:.4f} ({val_acc*100:.2f}%) | Val DER: {val_der:.4f} ({val_der*100:.2f}%)")
        
        # Track best for this model
        if val_der < best_der:
            best_der = val_der
            print(f"  ✅ Best {model_name} so far!")
            
            # Save if best overall
            if val_der < best_overall_der:
                best_overall_der = val_der
                best_model_name = model_name
                best_model_state = {
                    'model_name': model_name,
                    'model_state_dict': model.state_dict(),
                    'val_der': val_der,
                    'val_acc': val_acc,
                    'epoch': epoch,
                    'config': CONFIG,
                }
                torch.save(best_model_state, 'best_model.pt')
                print(f"  🏆 NEW BEST OVERALL MODEL!")

print(f"\n{'='*70}")
print(f"🎉 TRAINING COMPLETE!")
print(f"{'='*70}")
print(f"\n🏆 Best Model: {best_model_name}")
print(f"   DER: {best_overall_der:.4f} ({best_overall_der*100:.2f}%)")
print(f"   Accuracy: {(1-best_overall_der):.4f} ({(1-best_overall_der)*100:.2f}%)")
print(f"\n✓ Saved as: best_model.pt")

In [ ]:
# ============================================================================
# Test on Examples
# ============================================================================

print("\n" + "="*70)
print("🧪 TESTING ON EXAMPLES")
print("="*70)

# Load best model from CELL 17
checkpoint = torch.load('best_model.pt')
model_name = checkpoint['model_name']

# Recreate the correct model architecture
if model_name == 'BiLSTM':
    best_model = EnhancedBiLSTMDiacritizer(
        vocab_size=len(char_vocab),
        embedding_dim=CONFIG['embedding_dim'],
        positional_dim=CONFIG['positional_dim'],
        char_type_dim=CONFIG['char_type_dim'],
        hidden_dim=CONFIG['hidden_dim'],
        num_layers=CONFIG['num_layers'],
        num_classes=NUM_CLASSES,
        dropout=CONFIG['dropout'],
        num_heads=CONFIG['num_heads']
    ).to(device)
else:  # Transformer
    best_model = TransformerDiacritizer(
        vocab_size=len(char_vocab),
        embedding_dim=CONFIG['embedding_dim'],
        positional_dim=CONFIG['positional_dim'],
        char_type_dim=CONFIG['char_type_dim'],
        hidden_dim=CONFIG['hidden_dim'],
        num_layers=4,
        num_classes=NUM_CLASSES,
        dropout=CONFIG['dropout'],
        num_heads=CONFIG['num_heads']
    ).to(device)

best_model.load_state_dict(checkpoint['model_state_dict'])
print(f"\n✓ Loaded best model: {model_name}")
print(f"   Epoch: {checkpoint['epoch']+1}")
print(f"   DER: {checkpoint['val_der']:.4f}")
print(f"   Accuracy: {checkpoint['val_acc']:.4f}")

# Test examples (undiacritized text)
test_examples = [
    "ذهب الطفل الى المدرسة",
    "قرأ الكتاب بعناية",
    "يلعب الاطفال في الحديقة",
    "الشمس تشرق من الشرق",
    "احب اللغة العربية"
]

print("\n📝 Predictions:")
print("-" * 70)
for i, text in enumerate(test_examples, 1):
    pred = predict_diacritics(best_model, text, char_vocab, ID_TO_DIACRITIC, device)
    print(f"\n{i}. Input:  {text}")
    print(f"   Output: {pred}")

print("\n" + "-" * 70)

In [ ]:
# ============================================================================
# Save Summary
# ============================================================================

# Save training summary
summary = {
    'model_type': 'BiLSTM',
    'config': CONFIG,
    'best_der': float(best_der),
    'best_epoch': int(best_epoch),
    'total_epochs': CONFIG['num_epochs'],
    'train_losses': [float(x) for x in train_losses],
    'val_losses': [float(x) for x in val_losses],
    'val_ders': [float(x) for x in val_ders],
}

import json
with open('training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*70)
print("📁 FILES CREATED")
print("="*70)
print("\n✓ best_model.pt - Your trained model")
print("✓ training_summary.json - Training statistics")
print("\n💡 Download these from the Output tab →")
print("\n" + "="*70)
print("🎉 ALL DONE!")
print("="*70)
print("""
Next steps:
1. Download best_model.pt from Output tab
2. When you get test.txt, upload it and run prediction
3. For better results, try:
   - Increasing num_epochs to 15-20
   - Trying hidden_dim=768 or 1024
   - Experimenting with dropout values
   - Using the Transformer model instead
""")

In [ ]:
# # TODO: When you get test.txt, upload it and run this:

# print("Loading test data...")
# with open('/kaggle/input/test-data/test.txt', 'r', encoding='utf-8') as f:
#     test_texts = [line.strip() for line in f if line.strip()]

# print(f"Test samples: {len(test_texts)}")

# # Load best model
# checkpoint = torch.load('best_model.pt')
# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()

# # Generate predictions
# predictions = []
# for text in tqdm(test_texts, desc="Predicting"):
#     # Clean the text (remove existing diacritics if any)
#     cleaned = preprocessor.clean_text(text)
#     undiacritized, _ = preprocessor.extract_diacritics(cleaned)
    
#     # Predict
#     pred = predict_diacritics(model, undiacritized, char_vocab, ID_TO_DIACRITIC, device)
#     predictions.append(pred)

# # Save for Kaggle submission
# with open('submission.txt', 'w', encoding='utf-8') as f:
#     for pred in predictions:
#         f.write(pred + '\n')

# print("✅ Predictions saved to submission.txt")
# print("📥 Download from Output tab and submit to Kaggle competition!")